# Student Mental Health Binary Classification Model

### Objective
The goal of this notebook is to build and evaluate a machine learning model that predicts whether a student has depression (`Do you have Depression?`) based on their demographic details, academic status (CGPA, year of study), and other mental health indicators (Anxiety, Panic attacks). We will train a **Logistic Regression** model and measure its performance using key evaluation metrics.

In [1]:
# Import the required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load the student mental health dataset
df = pd.read_csv('Student_Mental_health.csv')

# Inspect the dataset dimensions and the first few rows
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Dataset shape: 101 rows, 11 columns


,Timestamp,gender,Age,What is your course?,Your current year of Study,What is your CGPA?,Marital status,Do you have Depression?,Do you have Anxiety?,Do you have Panic attack?,Did you seek any specialist for a treatment?
0,8/7/2020 12:02,Female,18.0,Engineering,year 1,3.00 - 3.49,No,Yes,No,Yes,No
1,8/7/2020 12:04,Male,21.0,Islamic education,year 2,3.00 - 3.49,No,No,Yes,No,No
2,8/7/2020 12:05,Male,19.0,BIT,Year 1,3.00 - 3.49,No,Yes,Yes,Yes,No
3,8/7/2020 12:06,Female,22.0,Laws,year 3,3.00 - 3.49,Yes,Yes,No,No,No
4,8/7/2020 12:13,Male,23.0,Mathemathics,year 4,3.00 - 3.49,No,No,No,No,No


### Data Preprocessing
To prepare this dataset for machine learning, we need to clean and transform the raw data:
1. **Drop high-cardinality/irrelevant columns**: The `Timestamp` is irrelevant for predictions, and `What is your course?` has $49$ unique categories for only $101$ records. Keeping it would cause our model to overfit.
2. **Standardize categorical text**: Year of Study and CGPA ranges contain trailing spaces or inconsistent casing (e.g., 'Year 1' vs 'year 1'). We will standardize these strings.
3. **Handle missing values**: Fill any missing values in the `Age` column with the median age.
4. **Target encoding**: Convert our binary target variable (`Do you have Depression?`) into numerical format ($1$ for Yes, $0$ for No).
5. **Feature encoding**: Convert remaining categorical features (like gender, anxiety, etc.) into binary numbers using one-hot encoding (`pd.get_dummies()`).

In [2]:
# 1. Drop the columns that are not useful or have too many unique values
df_cleaned = df.drop(columns=['Timestamp', 'What is your course?'])

# 2. Standardize casing and strip spaces for 'Your current year of Study' and 'What is your CGPA?'
df_cleaned['Your current year of Study'] = df_cleaned['Your current year of Study'].str.lower().str.strip()
df_cleaned['What is your CGPA?'] = df_cleaned['What is your CGPA?'].str.strip()

# 3. Fill the missing values in the 'Age' column using the median age of the dataset
median_age = df_cleaned['Age'].median()
df_cleaned['Age'] = df_cleaned['Age'].fillna(median_age)

# 4. Map the target column 'Do you have Depression?' to binary integers: Yes -> 1, No -> 0
target_col = 'Do you have Depression?'
df_cleaned[target_col] = df_cleaned[target_col].map({'Yes': 1, 'No': 0})

# 5. Separate our independent variables (X) and the target variable (y)
X = df_cleaned.drop(columns=[target_col])
y = df_cleaned[target_col]

# 6. Convert categorical feature columns to numeric dummy variables (0s and 1s)
# drop_first=True prevents multicollinearity by omitting the first category flag
X_encoded = pd.get_dummies(X, drop_first=True)

# Display the prepared feature columns
print("Processed Feature Columns:")
print(X_encoded.columns.tolist())

Processed Feature Columns:
['Age', 'gender_Male', 'Your current year of Study_year 2', 'Your current year of Study_year 3', 'Your current year of Study_year 4', 'What is your CGPA?_2.00 - 2.49', 'What is your CGPA?_2.50 - 2.99', 'What is your CGPA?_3.00 - 3.49', 'What is your CGPA?_3.50 - 4.00', 'Marital status_Yes', 'Do you have Anxiety?_Yes', 'Do you have Panic attack?_Yes', 'Did you seek any specialist for a treatment?_Yes']


### Splitting the Dataset
We split our dataset into a training set and a testing set using an $80/20$ split. This ensures we have $80\%$ of the data to train the model, and $20\%$ to evaluate how well it generalizes to new, unseen student records.

In [3]:
# Split the dataset into training (80%) and testing (20%) subsets
# A random_state of 42 is set to make the split reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, 
    test_size=0.2, 
    random_state=42
)

# Output dataset sizes
print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")

Training set: 80 samples
Testing set: 21 samples


### Model Training: Logistic Regression
We instantiate a **Logistic Regression** classifier. It is a highly efficient baseline model for binary classification, calculating the probability that an observation belongs to a given class (Depression: Yes or No).

In [4]:
# Instantiate the model with a high limit of iterations to ensure optimization converges
model = LogisticRegression(max_iter=1000)

# Train the model on the training data
model.fit(X_train, y_train)

# Generate predictions on the test dataset
predictions = model.predict(X_test)

print("Model training is complete, and test predictions have been generated!")

Model training is complete, and test predictions have been generated!


### Model Evaluation
To measure how well our classification model is performing, we will compute four fundamental metrics:
1. **Accuracy**: The percentage of overall predictions that are correct.
2. **Precision**: Out of all predicted positive cases (depression predicted as Yes), how many were actually correct?
3. **Recall**: Out of all actual positive cases in the test set, how many did the model successfully identify?
4. **F1-Score**: The harmonic mean of Precision and Recall, which balances both metrics.

In [5]:
# Calculate the classification evaluation metrics
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, zero_division=0)
recall = recall_score(y_test, predictions, zero_division=0)
f1 = f1_score(y_test, predictions, zero_division=0)

# Display the performance results
print("=========================================")
print("     LOGISTIC REGRESSION PERFORMANCE     ")
print("=========================================")
print(f"Accuracy:  {accuracy:.4f} (or {accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f} (or {precision*100:.2f}%)")
print(f"Recall:    {recall:.4f} (or {recall*100:.2f}%)")
print(f"F1-Score:  {f1:.4f} (or {f1*100:.2f}%)")
print("=========================================")

     LOGISTIC REGRESSION PERFORMANCE     
Accuracy:  0.7143 (or 71.43%)
Precision: 0.7500 (or 75.00%)
Recall:    0.3750 (or 37.50%)
F1-Score:  0.5000 (or 50.00%)


### Interpretation of Results

The Logistic Regression model achieved an accuracy of **$71.43\%$** on our test dataset.

* **Analysis of Performance**: Our precision score is **$75.00\%$**, which means when the model flags a student with depression, it is correct three-quarters of the time. However, our recall is lower at **$37.50\%$** (F1-score is **$50.00\%$**). This reveals that the model misses a significant number of actual depression cases, labeling them as negative. In a mental health diagnostic context, high recall is crucial to ensure students who need help are not missed.
* **Limitations**: The model's limitations likely stem from the small size of our dataset (only $101$ total samples) and class imbalance, as there are fewer positive depression cases than negative ones. Additionally, a linear decision boundary may not fully capture complex, non-linear interactions between social factors (like marital status) and academic performance (CGPA ranges).
* **Next Steps for Improvement**: To improve performance, we could try a different classification model, such as a **Decision Tree Classifier** or a **Random Forest Classifier**, which can capture non-linear interactions. We could also apply feature scaling (like `StandardScaler`) or use techniques to address class imbalance (such as adjusting class weights during model training).